In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src import config
from src.console import print_header, print_kv, print_status
from src.error_analysis import (attach_metadata, load_run_predictions, most_confident_errors,
                                plot_error_gallery, plot_utterance_diagnostics,
                                save_representative_utterance_artifacts)
from src.model_analysis import load_run_gate_weights
from src.results import build_per_class_metrics_table, select_analysis_run
from src.training.checkpoint import load_checkpoint
from src.training.data import load_manifest
from src.training.models import SEVERITY_MODEL_NAME, build_model
from src.training.metrics import compute_confusion_matrix, ordinal_mae
from src.training.reporting import save_confusion_matrix
from src.training.utils import resolve_device

config.ensure_directories()
TASK = "severity"

df_m6 = load_manifest()
RUN_NAME = select_analysis_run(task=TASK, metric="f1", preferred="severity_gated_fusion_three_branch")
print_header("Three-Branch Error Analysis")
print_kv("Analysis run", RUN_NAME or "none eligible yet -- run notebooks/03_training.ipynb first")

preds = None
if RUN_NAME is not None:
    try:
        preds = load_run_predictions(RUN_NAME)
        preds = attach_metadata(preds, df_m6)
    except FileNotFoundError as e:
        print_status(str(e), ok=False)

In [ ]:
if preds is not None:
    cm = compute_confusion_matrix(preds["y_true"].to_numpy(), preds["y_pred"].to_numpy(), TASK)
    cm_path = config.METRIC_FIGURE_DIR / f"{RUN_NAME}_confusion_matrix.png"
    save_confusion_matrix(cm_path, cm, TASK, title=f"{RUN_NAME} -- pooled confusion matrix")
    print_kv("Confusion matrix figure", cm_path)
    pd.DataFrame(cm, index=config.SEVERITY_CLASS_NAMES, columns=config.SEVERITY_CLASS_NAMES)

In [ ]:
per_class_table = None
if RUN_NAME is not None:
    try:
        per_class_table = build_per_class_metrics_table(RUN_NAME, task=TASK)
    except FileNotFoundError as e:
        print_status(str(e), ok=False)
per_class_table

In [ ]:
if preds is not None:
    rank_error = (preds["y_pred"] - preds["y_true"]).abs()
    print_kv("Ordinal MAE (pooled)", f"{ordinal_mae(preds['y_true'].to_numpy(), preds['y_pred'].to_numpy(), TASK):.4f}")
    print_kv("Rank-error distribution", dict(rank_error.value_counts().sort_index()))
    off_by_2_or_more = preds[rank_error >= 2]
    print_kv("Utterances off by >=2 severity ranks", len(off_by_2_or_more))
    off_by_2_or_more[["filename", "speaker_id", "y_true_label", "y_pred_label"]].head(10)

In [ ]:
if preds is not None:
    correct_sample = preds[preds["correct"].astype(bool)].sample(
        min(4, int(preds["correct"].sum())), random_state=config.DEFAULT_SEED)
    correct_dir = config.SIGNAL_FIGURE_DIR / RUN_NAME / "correct"
    for _, row in correct_sample.iterrows():
        plot_utterance_diagnostics(row, out_dir=correct_dir, show=False)
    print_kv("Correct-prediction gallery", f"{len(correct_sample)} figure(s) in {correct_dir}")

In [ ]:
if preds is not None:
    error_gallery_paths = plot_error_gallery(preds, RUN_NAME, n=8, show=False)
    top_errors = most_confident_errors(preds, n=8)
    top_errors[["filename", "speaker_id", "y_true_label", "y_pred_label", "confidence"]] if not top_errors.empty else top_errors

In [ ]:
device = resolve_device(None)
representative_model = None
ckpt_dir = config.CHECKPOINT_DIR / RUN_NAME if RUN_NAME is not None else None
if ckpt_dir is not None and ckpt_dir.exists():
    fold_dirs = sorted(p.parent.name for p in ckpt_dir.glob("*/best.pt"))
    if fold_dirs:
        representative_model = build_model(SEVERITY_MODEL_NAME, config.NUM_CLASSES[TASK], num_speakers=1).to(device)
        load_checkpoint(ckpt_dir / fold_dirs[0] / "best.pt", representative_model, map_location=str(device))
        representative_model.eval()

representative = save_representative_utterance_artifacts(
    df_m6, model=representative_model, device=device, n_per_class=2, seed=config.DEFAULT_SEED)
print_kv("Representative utterances (signal panel + branch embeddings + prediction)", len(representative))
representative[["Filename", "Speaker_ID", "Severity", "signal_figure"]]

In [ ]:
if RUN_NAME is not None:
    try:
        gate_df = load_run_gate_weights(RUN_NAME)
        if preds is not None:
            merged = gate_df.merge(preds[["filename", "y_true_label", "y_pred_label", "correct"]],
                                   on="filename", how="inner")
            print_kv("Prediction vs. actual severity (pooled)",
                     dict(merged.groupby(["y_true_label", "y_pred_label"]).size()))
            merged[["filename", "y_true_label", "y_pred_label", "correct",
                   "gate_learned", "gate_segmental", "gate_supra"]].head(15)
    except FileNotFoundError as e:
        print_status(str(e), ok=False)